In [1]:
import pandas as pd
from pathlib import Path

In [2]:
path_esios = Path("../data/silver/esios/esios_diario.parquet")
path_clima = Path("../data/silver/clima/clima.parquet")
path_festivos = Path("../data/silver/festivos/festivos.parquet")

In [3]:
gold_output = Path("../data/gold/dataset_final.parquet")
gold_output.parent.mkdir(parents=True, exist_ok=True)

In [4]:
df_esios = pd.read_parquet(path_esios)
df_clima = pd.read_parquet(path_clima)
df_festivos = pd.read_parquet(path_festivos)

In [5]:
df_esios["fecha"] = pd.to_datetime(df_esios["fecha"], errors="coerce").dt.tz_localize(None)
df_clima["fecha"] = pd.to_datetime(df_clima["fecha"], errors="coerce")
df_festivos["fecha"] = pd.to_datetime(df_festivos["fecha"], errors="coerce")

In [6]:
columnas_festivos = [col for col in ["fecha", "festivo_nombre", "festivo_nacional", "es_festivo"] if col in df_festivos.columns]
df_festivos = df_festivos[columnas_festivos].copy()

In [7]:
df_festivos = df_festivos.drop_duplicates(subset=["fecha"])

In [8]:
df_gold = df_esios.merge(df_clima, on="fecha", how="left")
df_gold = df_gold.merge(df_festivos, on="fecha", how="left")

In [9]:
print(df_esios["fecha"].dtype)
print(df_clima["fecha"].dtype)
print(df_festivos["fecha"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [10]:
if "es_festivo" in df_gold.columns:
    df_gold["es_festivo"] = df_gold["es_festivo"].fillna(0).astype(int)

if "festivo_nacional" in df_gold.columns:
    df_gold["festivo_nacional"] = df_gold["festivo_nacional"].fillna(0).astype(int)

if "festivo_nombre" in df_gold.columns:
    df_gold["festivo_nombre"] = df_gold["festivo_nombre"].fillna("No festivo")

In [11]:
df_gold["dia_semana"] = df_gold["fecha"].dt.dayofweek     # lunes=0
df_gold["nombre_dia"] = df_gold["fecha"].dt.day_name()
df_gold["fin_de_semana"] = df_gold["dia_semana"].isin([5, 6]).astype(int)
df_gold["mes"] = df_gold["fecha"].dt.month
df_gold["trimestre"] = df_gold["fecha"].dt.quarter
df_gold["dia_del_ano"] = df_gold["fecha"].dt.dayofyear

In [12]:
if "temperatura_media" in df_gold.columns and "es_festivo" in df_gold.columns:
    df_gold["temp_x_festivo"] = df_gold["temperatura_media"] * df_gold["es_festivo"]

In [13]:
df_gold = df_gold.sort_values("fecha").reset_index(drop=True)

In [14]:
print(df_gold.info())
print(df_gold.isnull().sum())
display(df_gold.head(12))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   fecha                       731 non-null    datetime64[ns]
 1   eolica                      731 non-null    float64       
 2   generacion_renovable_total  731 non-null    float64       
 3   hidraulica                  731 non-null    float64       
 4   hidroeolica                 730 non-null    float64       
 5   otras_renovables            731 non-null    float64       
 6   residuos_renovables         731 non-null    float64       
 7   solar_fotovoltaica          731 non-null    float64       
 8   solar_termica               731 non-null    float64       
 9   year                        731 non-null    int32         
 10  month                       731 non-null    int32         
 11  day                         731 non-null    int32         

,fecha,eolica,generacion_renovable_total,hidraulica,hidroeolica,otras_renovables,residuos_renovables,solar_fotovoltaica,solar_termica,year,...,festivo_nombre,festivo_nacional,es_festivo,dia_semana,nombre_dia,fin_de_semana,mes,trimestre,dia_del_ano,temp_x_festivo
0,2024-01-01,159629.924,296067.6560,58381.148,3.650,6171.128,2470.3430,66642.505,2768.958,2024,...,Año Nuevo,1,1,0,Monday,0,1,1,1,9.207096
1,2024-01-02,307120.989,436751.7150,68004.167,2.920,6820.630,2427.9510,49615.015,2760.043,2024,...,No festivo,0,0,1,Tuesday,0,1,1,2,0.000000
2,2024-01-03,284990.434,426342.5480,93862.430,3.037,7570.068,2412.0980,36440.805,1063.676,2024,...,No festivo,0,0,2,Wednesday,0,1,1,3,0.000000
3,2024-01-04,138827.131,315220.3270,139605.013,9.976,8209.237,2548.5420,26019.065,1.363,2024,...,No festivo,0,0,3,Thursday,0,1,1,4,0.000000
4,2024-01-05,306634.568,464836.5590,83467.494,7.722,8140.782,2421.0290,61906.689,2258.275,2024,...,No festivo,0,0,4,Friday,0,1,1,5,0.000000
5,2024-01-06,306142.135,475366.9135,77899.805,2.651,7577.274,2416.8615,78224.394,3103.793,2024,...,Día de Reyes / Epifanía del Señor,1,1,5,Saturday,1,1,1,6,7.321650
6,2024-01-07,264209.990,454758.5325,89119.710,3.196,8387.540,2264.7795,85346.083,5427.234,2024,...,No festivo,0,0,6,Sunday,1,1,1,7,0.000000
7,2024-01-08,194169.846,417775.3860,145217.971,5.787,9143.409,2207.9860,65444.285,1586.102,2024,...,No festivo,0,0,0,Monday,0,1,1,8,0.000000
8,2024-01-09,26139.440,247723.7260,172224.091,6.182,9350.501,2183.5990,37575.466,244.447,2024,...,No festivo,0,0,1,Tuesday,0,1,1,9,0.000000
9,2024-01-10,91726.101,300504.8970,162337.094,17.048,9595.661,2524.5930,33361.581,942.819,2024,...,No festivo,0,0,2,Wednesday,0,1,1,10,0.000000


In [15]:
df_gold.to_parquet("../data/gold/dataset_final.parquet", index=False)